# Streaming

<img src="./assets/LC_streaming.png" width="400">

Streaming reduces the latency between generating data and the user receiving it.
There are two types frequently used with Agents:

Let's start by setting up a basic agent to demonstrate different streaming approaches.


In [1]:
import * as setup from "./setup.ts";
import { createAgent } from "langchain";
import { ChatGoogle } from "@langchain/google";

const model =new ChatGoogle({
    model: "gemini-3.5-flash-lite",
});

const agent = createAgent({
    model,
    systemPrompt: "You are a full-stack comedian",
})

Now that we have our agent configured, let's first see how it works without streaming.


## No Streaming (invoke)

In [2]:
import { HumanMessage } from "langchain";

const result = await agent.invoke({
    messages: [new HumanMessage("Tell me a joke")]
})

console.log(result.messages.at(-1).content)

My therapist told me I have a preoccupation with vengeance. 

Well, we'll see about that.


Notice how we had to wait for the complete response. Let's improve the user experience with streaming.


## Streaming
### `value`

In [3]:
const stream = await agent.stream({
    messages: [new HumanMessage("Tell me a joke")],
}, {
    streamMode: "values",
})

for await (const step of stream) {
    console.log(step.messages.at(-1).content)
}

Tell me a joke
My girlfriend told me she wanted someone who’s really good at troubleshooting and never gives up.

So naturally, I deployed a microservice, ignored all the upstream dependencies, and now we’re living in an endless loop of dependency hell. 

The relationship is completely unresponsive, but hey—at least my local environment is running fine.


The `values` mode streams the entire state at each step. This is useful when you want to see the full message history as it evolves.

### `messages`

For a more granular experience, `messages` mode streams individual message chunks as they're generated - perfect for real-time token-by-token display.


In [5]:
const stream = await agent.stream({
    messages: [new HumanMessage("Tell me a joke")],
}, {
    streamMode: "messages",
})

for await (const [message, metadata] of stream) {
    console.log(`[${metadata.langgraph_node}]: ${message.content}`)
}



[model_request]: A frontend
[model_request]:  developer and a backend developer walk into a bar.

The frontend developer immediately starts
[model_request]:  complaining, "Man, this user experience is terrible. The lighting is asynchronous, the tables take three business days to render, and I
[model_request]: 'm pretty sure the bartender is running on Internet Explorer."

The backend developer rolls his eyes, orders a straight shot
[model_request]:  of whiskey, and says, "Relax, it works fine on my machine. Besides, nobody cares what it looks like as
[model_request]:  long as the database holds up."

Halfway through their drinks, a DevOps engineer bursts through the front door screaming, "EVERY
[model_request]: THING IS ON FIRE! THE CONTAINER IS LEAKING, THE LOAD BALANCER IS CRYING, AND PRODUCTION IS DOWN
[model_request]: !"

The backend dev panics, drops his glass, and starts frantically typing on his laptop. 

The frontend dev doesn
[model_request]: 't even look up from his phone. He

ReferenceError: message is not defined

Here's a more dramatic example - streaming a poem token by token creates a typewriter effect:


In [9]:
await Deno.jupyter.broadcast("display_data", {
    data: { "text/markdown": "🤔" },
    metadata: {},
    transient: { display_id: "progress" }
});

const stream = await agent.stream({
    messages: [new HumanMessage("Write me a poem.")],
}, {
    streamMode: "messages",
})

let content = "";
const i = setInterval(async () => {
    Deno.jupyter.broadcast("update_display_data", {
        data: { "text/markdown": content },
        metadata: {},
        transient: { display_id: "progress" }
    });
}, 1)

for await (const [message, metadata] of stream) {
    content += message.content
}

clearInterval(i)

I sit at my desk, face illuminated by the cold, blue light of three monitors.
Coffee cup empty. Git status: dirty.
I am a Full-Stack Comedian,
And today, my architecture is failing.

**Front-End (The Setup):**
I try to make the audience laugh.
I render HTML smiles, CSS wit.
I’m responsive! I scale to all viewports!
I look great on mobile, I swear!
I put on a hat for the UX—User Experience—
Hoping they’ll click my jokes and stay a while.
I drop a CSS grid framework of charisma,
And I wait.
A single clatter of keys. 
Console log: *“Crickets found.”*
Undefined is not a function of joy.

**Back-End (The Punchline):**
So I panic, and I route the traffic to the server.
Node.js, Express, middleware of pure self-doubt.
I query the database of my childhood traumas
Looking for a primary key that still hurts.
SQL injection of pure anxiety.
*SELECT * FROM audience WHERE expression = 'amused';*
Result: `0 rows affected.`

*Error 500: Internal Server Crying.*

My database connection timed out.
My AWS bill is three thousand dollars 
Because I accidentally infinite-looped a knock-knock joke
Into the cloud.

**DevOps (The Heckler):**
Now the hecklers arrive, deployed via Docker containers.
"Your syntax is garbage!" they yell.
"You forgot a semicolon in your premise!"
I try to run `git push origin mental-breakdown`,
But there are merge conflicts.
Always merge conflicts.
My therapist is my git rebase—
Painful, destructive, and usually ends with me force-quitting.

**Full-Stack Summary:**
I write jokes for the browser,
I query tears from the cloud,
And at the end of the sprint,
I push to production without testing,
Hoping nobody notices 
That the backend is just a terrified intern
And the front-end is screaming.

Thank you. 
Please tip your waitstaff, 
And if anyone knows how to debug a life,
Please open a pull request. 
I’m begging you.

## Tools can stream too!
Streaming generally means delivering information to the user before the final result is ready. There are many cases where this is useful. A stream writer allows you to easily stream `custom` data from sources you create.

In [12]:
import * as setup from "./setup.ts";
import { z } from "zod";
import { createAgent, tool, type Runtime } from "langchain";
import { AsyncLocalStorageProviderSingleton } from "npm:@langchain/core/singletons";

const getWeather = tool(({ city }, runtime: Runtime) => {
    runtime.writer(`Looking up data for city: ${city}`);
    runtime.writer(`Acquired data for city: ${city}`);
    return `It's always sunny in ${city}`
}, {
    name: "get_weather",
    description: "Get weather for a given city.",
    schema: z.object({
        city: z.string()
    })
});

const toolCallingAgent = createAgent({
    model,
    systemPrompt: "You are a helpful assistant that can call tools.",
    tools: [getWeather]
})

// Note: Deno Jupyter requires wrapping stream() in a clean async context
const stream = await AsyncLocalStorageProviderSingleton.runWithConfig(
    {},
    () => toolCallingAgent.stream({
        messages: "What's the weather in Bihar,India?",
    }, {
        streamMode: ["values", "custom"],
    })
);

for await (const [type, stateOrCustomEvent] of stream) {
    if (type === "values") {
        displayMessage(stateOrCustomEvent.messages.at(-1))
    } else if (type === "custom") {
        displayMessage({
            type,
            content: stateOrCustomEvent
        })
    }
}


┌────────────────────────────────────────────────────────────┐
│ 👤 HUMAN MESSAGE                                           │
└────────────────────────────────────────────────────────────┘
What's the weather in Bihar,India?

┌────────────────────────────────────────────────────────────┐
│ 🤖 AI MESSAGE                                              │
└────────────────────────────────────────────────────────────┘
[
  {
    thoughtSignature: "EmAKXgFpFH0Tzc6hCfs4YyBpyDz4itsf/NdUZ67SWowoDBC8jEvvJ5yJ093dQ5e+hstJhEd2KhM0LzfgUd5k34C2so4oRUCp21DldUQEMoLgeb+KobHBOZRsruab1V1ROLc=",
    type: "functionCall",
    functionCall: {
      name: "get_weather",
      args: { city: "Bihar, India" },
      id: "call_8049383"
    }
  }
]

┌────────────────────────────────────────────────────────────┐
│ 💡 CUSTOM MESSAGE                                          │
└────────────────────────────────────────────────────────────┘
Looking up data for city: Bihar, India

┌─────────────────────────────────────────────